<a href="https://colab.research.google.com/github/ankurdev1-drth/Deep_Learning-/blob/main/02_Training_Deep_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [1]:
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import torch

layer = nn.Linear(40, 10)
layer.weight.data *= 6 ** 0.5  # kaiming init (or 3 ** 0.5 for LeCun)
torch.zero_(layer.bias.data)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [3]:
layer.weight.shape

torch.Size([10, 40])

In [4]:
layer.bias.shape

torch.Size([10])

In [5]:
# another way to for initialization
nn.init.kaiming_uniform_(layer.weight)
nn.init.zeros_(layer.bias)

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

This is clearer and less error prone as compared to the first approach

For applying initialization to the weights of every `nn.Linear` layer in a model the simples option is to write a little function that takes a module, checks whether it's an instance of the `nn.Linear` class, and if so , applies the desired initialization function to its weights. and the function can be then applied using the `apply()` method

In [6]:
def use_he_init(module):
  if isinstance(module, nn.Linear):
    nn.init.kaiming_uniform_(module.weight)
    nn.init.zeros_(module.bias)

module = nn.Sequential(
    nn.Linear(40, 10),
    nn.ReLU(),
    nn.Linear(10, 1),
    nn.ReLU()
    )
module.apply(use_he_init)

Sequential(
  (0): Linear(in_features=40, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=1, bias=True)
  (3): ReLU()
)

### Leaky ReLU

In [7]:
alpha = 0.2
model = nn.Sequential(
    nn.Linear(50, 40), #0
    nn.LeakyReLU(negative_slope=alpha), #1
    nn.Linear(40, 20), #2
    nn.LeakyReLU(negative_slope=alpha) #3

)
nn.init.kaiming_uniform_(model[0].weight, alpha,
nonlinearity="leaky_relu")
nn.init.kaiming_uniform_(model[2].weight, alpha,nonlinearity="leaky_relu")

Parameter containing:
tensor([[-0.2638,  0.3082, -0.0620, -0.0147, -0.2633,  0.3209,  0.2701,  0.3276,
         -0.2923, -0.3498, -0.2855, -0.2711, -0.3528, -0.0100,  0.2500, -0.3172,
         -0.0447, -0.3319,  0.2178, -0.2796, -0.0013, -0.0691,  0.1415, -0.3298,
          0.3094,  0.2139, -0.3542, -0.0646,  0.1140,  0.0653, -0.2393,  0.2994,
         -0.3660, -0.2795,  0.1689,  0.0636, -0.0826,  0.0859, -0.2005,  0.0517],
        [ 0.2315,  0.1634, -0.0982,  0.0776, -0.3026, -0.2901,  0.0967,  0.1444,
          0.3554,  0.2572,  0.3103,  0.0627,  0.1697, -0.2961, -0.3016, -0.0207,
         -0.2634, -0.2433, -0.1100, -0.3768, -0.3267, -0.0453,  0.1395,  0.0915,
          0.0188, -0.0168, -0.2067,  0.3645,  0.1400,  0.1727, -0.3357, -0.3349,
         -0.0143,  0.3472, -0.1254, -0.2016, -0.1220, -0.2980,  0.0796, -0.0419],
        [-0.1858,  0.1750,  0.0839,  0.0618, -0.0494,  0.1414,  0.0466,  0.0879,
         -0.3168,  0.1001, -0.3208, -0.1495,  0.2596, -0.3062, -0.2638, -0.2745,
    

### Batch Normalization

In [8]:
model = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(1 * 28 * 28),
    nn.Linear(1 * 28 * 28,  300),
    nn.ReLU(),
    nn.BatchNorm1d(300),
    nn.Linear(300, 100),
    nn.ReLU(),
    nn.BatchNorm1d(100),
    nn.Linear(100, 10)
)

In [9]:
dict(model[1].named_parameters()).keys()

dict_keys(['weight', 'bias'])

In [10]:
dict(model[1].named_buffers()).keys()

dict_keys(['running_mean', 'running_var', 'num_batches_tracked'])

In [11]:
dict(model[1].named_children()).keys()

dict_keys([])

In [12]:
dict(model[1].named_modules()).keys()

dict_keys([''])

Note:
- if BN layers before the activation functions, we can also remove the bias term from the previous `nn.Linear` layers by setting the `bias` hyperparameter to `False`.

In [13]:
Model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 300, bias=False),
    nn.BatchNorm1d(300),
    nn.ReLU(),
    nn.Linear(300, 100, bias=False),
    nn.BatchNorm1d(100),
    nn.ReLU(),
    nn.Linear(100, 10)
)